# The story of the perturbation retrievals...



In [1]:
# conda env 'vis2_clone'
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import seaborn as sns

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.ExperimentSet import ExperimentSet
from src.Retrieval import Retrieval

In [3]:
map_exp_name = {
    'default': {'name':'Default', 'order':1},
    '0p7': {'name':'Bright Surface', 'order':2},
    '1p3': {'name':'Dark Surface', 'order':3},
    '4p0': {'name':'High Surf Wind', 'order':4},
    '0p150': {'name':'Low AOD', 'order':5},
    '0p250': {'name':'High AOD', 'order':6},
    'mix22': {'name':'Aerosol Mixture 22', 'order':7},
    'mix54': {'name':'Aerosol Mixture 54', 'order':8},
    'unconstrained': {'name':'Unconstrained', 'order':9},
    'LCL_CTH': {'name':'LCL + CTH', 'order':10},
    'unconstrained_darksurf': {'name':'Unc. Dark Surf.', 'order':11},
    'periodic': {'name':'Periodic', 'order':12},
    'slow': {'name':'Slow Cloud Motion', 'order':13},
    'fast': {'name':'Fast Cloud Motion', 'order':14},
    'slow_unconstrained': {'name':'Slow Cloud Motion Unc.', 'order':15},
    'fast_unconstrained': {'name':'Fast Cloud Motion Unc.', 'order':16},
    'fast_unconstrained_restart': {'name':'Fast Unc. Restart', 'order':17}
}

In [4]:
EXPERIMENT_SET = PROJECT_ROOT / 'experiments' / 'CAMP2Ex_RF17' / 'V09'
V08 = ExperimentSet(EXPERIMENT_SET)
V08.set_retrieval_list()
V08.load_retrievals(iteration=-1, extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])

In [5]:
extinction_all = V08.to_dataframe(0.67, 'cloud', 'extinction', name_map=map_exp_name)
bins = np.arange(0.350, 1.251, 0.1)
extinction_all['altitude_bin'] = pd.cut(extinction_all['z'], bins=bins)
cth_all = V08.to_dataframe(0.67, 'cloud', 'cth', name_map=map_exp_name)
cbh_all = V08.to_dataframe(0.67, 'cloud', 'cbh', name_map=map_exp_name)
geometric_thickness_all = V08.to_dataframe(0.67, 'cloud', 'geometric_thickness', name_map=map_exp_name)

# Distributions

These variables have a distribution over the domain. They could be 3D or 2D.

- Cloud top height
- Cloud base height
- Cloud geometric thickness
- Extinction

In [6]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Default',
#             extinction_all['name'] == 'LCL + CTH',
#             extinction_all['name'] == 'Unconstrained'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [7]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Periodic',
#             extinction_all['name'] == 'Default'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [8]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Bright Surface',
#             extinction_all['name'] == 'Default',
#             extinction_all['name'] == 'Dark Surface'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [9]:
# f, ax = plt.subplots(1, 1, figsize=(8, 10))
# sns.boxplot(
#     data=extinction_all.loc[
#         np.logical_or.reduce((
#             extinction_all['name'] == 'Low AOD',
#             extinction_all['name'] == 'Default',
#             extinction_all['name'] == 'High AOD'
#         ))
#     ], 
#     x='extinction', 
#     y='altitude_bin', 
#     hue='name', 
#     whis=(0, 100), 
#     ax=ax
# )

In [10]:
# f, ax = plt.subplots(1, 1, figsize=(8, 20))
# sns.boxplot(data=extinction_all, x='extinction', y='altitude_bin', hue='name', whis=(0, 100), ax=ax)

In [11]:
# f, ax = plt.subplots(1, 1, figsize=(8, 20))
# sns.boxplot(data=extinction_all, x='extinction', y='name', hue='altitude_bin', whis=(0, 100), ax=ax)

In [12]:
# # build an order of display names consistent with the existing 'order' column
# order = extinction_all[['name', 'order']].drop_duplicates().sort_values('order')['name'].tolist()
# sns.boxplot(data=extinction_all, x='extinction', y='name', order=order, whis=(0, 100))

In [13]:
# f, ax = plt.subplots(1, 1, figsize=(8, 20))
# sns.violinplot(data=cth_all, x='cth', y='name', order=order, ax=ax)

In [14]:
# sns.boxplot(data=cth_all, x='cth', y='name', order=order)

In [15]:
# sns.boxplot(data=geometric_thickness_all, x='geometric_thickness', y='name', order=order)

# Extrinsic properteis: second set

The second set of extrinsic properties are the extrinsic properties that have a single statistic over the whole cloud domain. These include

- Mean & standard deviation CTH
- Mean & standard deviation CBH
- Mean & standard deviation geometric thickness
- Cloud volume
- Cloud fraction

In [16]:
cth_all.groupby(['name', 'order']).cth.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,0.685435,0.164110,0.4,1.1,0.6
2,Bright Surface,2,0.694342,0.164124,0.4,1.1,0.6
3,Dark Surface,3,0.669931,0.161400,0.4,1.1,0.6
9,High Surf Wind,4,0.658885,0.155448,0.4,1.1,0.6
11,Low AOD,5,0.675316,0.158829,0.4,1.1,0.6
8,High AOD,6,0.679066,0.163756,0.4,1.1,0.6
0,Aerosol Mixture 22,7,0.684975,0.163413,0.4,1.1,0.6
1,Aerosol Mixture 54,8,0.680423,0.162308,0.4,1.1,0.6
15,Unconstrained,9,1.069142,0.335993,0.1,1.4,1.1
10,LCL + CTH,10,0.681054,0.162208,0.4,1.1,0.6


In [17]:
cbh_all.groupby(['name', 'order']).cbh.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,0.501469,0.133662,0.4,1.1,0.5
2,Bright Surface,2,0.518246,0.143483,0.4,1.1,0.5
3,Dark Surface,3,0.516820,0.145816,0.4,1.1,0.5
9,High Surf Wind,4,0.477221,0.120708,0.4,1.1,0.4
11,Low AOD,5,0.480564,0.121563,0.4,1.1,0.4
8,High AOD,6,0.547139,0.155902,0.4,1.1,0.5
0,Aerosol Mixture 22,7,0.488300,0.123446,0.4,1.1,0.4
1,Aerosol Mixture 54,8,0.514507,0.139719,0.4,1.1,0.5
15,Unconstrained,9,0.521230,0.367687,0.0,1.4,0.5
10,LCL + CTH,10,0.514304,0.140427,0.4,1.1,0.5


In [18]:
geometric_thickness_all.groupby(['name', 'order']).geometric_thickness.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,0.183966,0.166200,0.0,0.7,0.20
2,Bright Surface,2,0.176096,0.169761,0.0,0.7,0.20
3,Dark Surface,3,0.153111,0.158866,0.0,0.7,0.10
9,High Surf Wind,4,0.181664,0.160251,0.0,0.7,0.20
11,Low AOD,5,0.194752,0.165660,0.0,0.7,0.20
8,High AOD,6,0.131928,0.157004,0.0,0.7,0.10
0,Aerosol Mixture 22,7,0.196675,0.168042,0.0,0.7,0.20
1,Aerosol Mixture 54,8,0.165915,0.164797,0.0,0.7,0.10
15,Unconstrained,9,0.547912,0.411636,0.0,1.4,0.45
10,LCL + CTH,10,0.166750,0.164152,0.0,0.7,0.10


In [19]:
extinction_all.groupby(['name', 'order']).extinction.agg(['mean', 'std', 'min', 'max', 'median']).reset_index().sort_values('order')

,name,order,mean,std,min,max,median
4,Default,1,7.277551,7.308989,0.300742,55.747046,4.750293
2,Bright Surface,2,7.820793,7.766404,0.305300,57.985802,5.243794
3,Dark Surface,3,8.967420,10.617444,0.307331,116.496951,5.056492
9,High Surf Wind,4,7.331772,8.123668,0.307404,77.964491,4.404240
11,Low AOD,5,6.543920,7.117048,0.301894,61.152506,4.029326
8,High AOD,6,10.890787,11.750220,0.308475,104.265077,6.660310
0,Aerosol Mixture 22,7,6.998992,6.744681,0.302474,47.678334,4.749386
1,Aerosol Mixture 54,8,8.825547,9.465522,0.300126,78.236337,5.542268
15,Unconstrained,9,5.025467,5.018172,0.303112,39.958458,3.332193
10,LCL + CTH,10,8.187404,8.621158,0.304562,74.075831,5.272225


In [20]:
brf_ths = {
    'DF': 0.120,
    'CF': 0.076,
    'BF': 0.054,
    'AF': 0.044,
    'AN': 0.040,
    'AA': 0.045,
    'BA': 0.052,
    'CA': 0.068,
    'DA': 0.092
}
for rpath in V08.retrievals:
    r = V08.retrievals[rpath]
    r.add_threshold_mask(r.obs, 'MISR', 'threshold_mask', 'brf', brf_ths)

In [21]:
def map_df(df, map_exp_name, ES):
    df = df.copy(deep=True)
    df['Retrieval Name'] = ''
    df['Order'] = 0
    for i in range(len(df)):
        new_name, order = map_exp_name[ES.retrievals[Path(df.reset_index().loc[i, 'retrieval'])].exp_name]
        df.loc[df.index == df.reset_index().loc[i, 'retrieval'], 'Retrieval Name'] = new_name
        df.loc[df.index == df.reset_index().loc[i, 'retrieval'], 'Order'] = order
    return df.reset_index().set_index('Order')

In [26]:
map_exp_name = {
    'default': ('Default', 1),
    '0p7': ('Bright Surface', 2),
    '1p3': ('Dark Surface', 3),
    '4p0': ('High Surf Wind', 4),
    '0p150': ('Low AOD', 5),
    '0p250': ('High AOD', 6),
    'mix22': ('Aerosol Mixture 22', 7),
    'mix54': ('Aerosol Mixture 54', 8),
    'unconstrained': ('Unconstrained', 9),
    'LCL_CTH': ('LCL + CTH', 10),
    'unconstrained_darksurf': ('Unc. Dark Surf.', 11),
    'periodic': ('Periodic', 12),
    'slow': ('Slow Cloud Motion', 13),
    'fast': ('Fast Cloud Motion', 14),
    'slow_unconstrained': ('Slow Cloud Motion Unc.', 15),
    'fast_unconstrained': ('Fast Cloud Motion Unc.', 16),
    'fast_unconstrained_restart': ('Fast Unc. Restart', 17)
}

In [27]:
V08_all = ExperimentSet(EXPERIMENT_SET)
V08_all.set_retrieval_list()
V08_all.load_retrievals(iteration=-1, extinction_th=[0.0, np.inf], tau_th=[0.0, np.inf])

In [28]:
def misr_S_1(y):
    return np.diag(1 / (0.04 * y)**2)
mask_conds = {
    # 'tau0.3mask': (1, 1),
    'threshold_mask': (1, 1),
    'ray_containment': (2, 2)
}
chi2 = V08.chi2(misr_S_1, variable='I', mask_conds=mask_conds)

stats_th = V08.cloud_stats(0.67, 'cloud')
stats_all = V08_all.cloud_stats(0.67, 'cloud')
stats_diff = stats_all - stats_th

chi2['Masked Cloud Volume'] = stats_diff['volume']
chi2['Unmasked Cloud Volume'] = stats_th['volume']

ref_key = Path('/data/gdi/b/johntl3/LundstromPub2025/experiments/CAMP2Ex_RF17/V09/default')
stats_th_ref = V08.compare_to(ref_key, 0.67, 'cloud', extinction_th=[0.3, np.inf], tau_th=[0.3, np.inf])

stats_th['Relative Bias (%)'] = stats_th_ref['relative bias']
stats_th['Relative Difference (%)'] = stats_th_ref['relative difference']

stats_th['Default CF'] = float(stats_th.loc[stats_th.index==str(ref_key), 'cloud fraction'])
stats_th['Default tau bar'] = float(stats_th.loc[stats_th.index==str(ref_key), 'tau bar'])
stats_th['Default volume'] = float(stats_th.loc[stats_th.index==str(ref_key), 'volume'])
stats_th['Rel CF Diff'] = 100*(stats_th['cloud fraction'] - stats_th['Default CF']) / stats_th['Default CF']
stats_th['Rel tau bar Diff'] = 100*(stats_th['tau bar'] - stats_th['Default tau bar']) / stats_th['Default tau bar']
stats_th['Rel volume Diff'] = 100*(stats_th['volume'] - stats_th['Default volume']) / stats_th['Default volume']


/tmp/ipykernel_1589553/22534209.py:23: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  stats_th['Default CF'] = float(stats_th.loc[stats_th.index==str(ref_key), 'cloud fraction'])
/tmp/ipykernel_1589553/22534209.py:24: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  stats_th['Default tau bar'] = float(stats_th.loc[stats_th.index==str(ref_key), 'tau bar'])
/tmp/ipykernel_1589553/22534209.py:25: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  stats_th['Default volume'] = float(stats_th.loc[stats_th.index==str(ref_key), 'volume'])


In [29]:
map_df(chi2, map_exp_name, V08).loc[:, ['Retrieval Name', 'chi2', 'Masked Cloud Volume', 'Unmasked Cloud Volume']]

,Retrieval Name,chi2,Masked Cloud Volume,Unmasked Cloud Volume
Order,,,,
1,Default,8.984105,7.267562,16.584562500000008
5,Low AOD,7.887847,15.041812,21.008625000000006
6,High AOD,13.206345,3.425812,10.375750000000004
7,Aerosol Mixture 22,8.790481,9.067438,17.401312500000007
8,Aerosol Mixture 54,9.914643,6.38275,13.060437500000003
14,Fast Cloud Motion,11.936640,8.038938,17.476937500000005
16,Fast Cloud Motion Unc.,14.999378,14.565375,24.68021875000001
13,Slow Cloud Motion,12.219415,7.759125,13.839375000000006
15,Slow Cloud Motion Unc.,10.997282,13.321344,28.752625000000016


In [30]:
map_df(stats_th, map_exp_name, V08).loc[:, ['Retrieval Name', 'Relative Bias (%)', 'Relative Difference (%)', 'Default CF', 'cloud fraction', 'Rel CF Diff', 'Default tau bar', 'tau bar', 'Rel tau bar Diff', 'Default volume', 'volume', 'Rel volume Diff']]

,Retrieval Name,Relative Bias (%),Relative Difference (%),Default CF,cloud fraction,Rel CF Diff,Default tau bar,tau bar,Rel tau bar Diff,Default volume,volume,Rel volume Diff
Order,,,,,,,,,,,,
1,Default,0.000000,0.000000,0.193373,0.19337278106508876,0.0,1.953448,1.9534478529123338,0.0,16.584563,16.584562500000008,0.0
5,Low AOD,13.919377,0.431579,0.193373,0.24355029585798815,25.948592,1.953448,1.76666750692714,-9.561573,16.584563,21.008625000000006,26.675787
6,High AOD,-6.327177,1.871385,0.193373,0.15715976331360948,-18.72705,1.953448,2.250325361682486,15.197616,16.584563,10.375750000000004,-37.437301
7,Aerosol Mixture 22,0.873969,0.422663,0.193373,0.19218934911242602,-0.611995,1.953448,1.9833351036654323,1.529974,16.584563,17.401312500000007,4.924761
8,Aerosol Mixture 54,-4.429735,1.055002,0.193373,0.1680473372781065,-13.096695,1.953448,2.1467210113317643,9.89395,16.584563,13.060437500000003,-21.24943
14,Fast Cloud Motion,-0.243122,1.687310,0.193373,0.19692307692307692,1.835985,1.953448,1.9139625710231105,-2.021312,16.584563,17.476937500000005,5.380757
16,Fast Cloud Motion Unc.,24.515612,3.065969,0.193373,0.21349112426035502,10.403917,1.953448,2.0001243797150354,2.389443,16.584563,24.68021875000001,48.814409
13,Slow Cloud Motion,0.412801,1.730041,0.193373,0.20023668639053255,3.549572,1.953448,1.8939153746260429,-3.047559,16.584563,13.839375000000006,-16.552668
15,Slow Cloud Motion Unc.,29.507408,3.275981,0.193373,0.2144378698224852,10.893513,1.953448,2.0924875238409615,7.117655,16.584563,28.752625000000016,73.369813
